# Stage D (Part 1) — Build Static Basin Attributes

**Purpose**
Create a clean, modeling-ready table of **static geospatial attributes** per basin and save it for reuse in later stages.

---

## What this notebook does

1. Load processed sources:

   * `data/boundaries/processed/basin_shape_stats.csv`
   * `data/boundaries/processed/basin_dem_acc_stats.csv`
2. Normalize basin names, merge the two sources, and attach `basin_id` via
   `data/boundaries/processed/basin_lookup.csv`.
3. Compute **derived features** (compactness, elevation range, coefficients of variation, etc.).
4. Keep stabilized flow-accumulation signals (`log1p(acc_mean)`, `log1p(acc_max)`) and two slope-fraction columns as generic `pct_slope_gt_a` and `pct_slope_gt_b` (thresholds unknown).
5. Support **plug-in extras**: any CSV/Parquet placed in `data/modeling/static/extra/` (with `basin_id` or `basin_name`) is auto-merged.
6. Write the final static table and a small provenance file.

---

## Inputs

* `data/boundaries/processed/basin_shape_stats.csv`  *(centroid, area, perimeter)*
* `data/boundaries/processed/basin_dem_acc_stats.csv`  *(DEM stats, slope, relief, accumulation)*
* `data/boundaries/processed/basin_lookup.csv`  *(maps `basin_name ↔ basin_id`)*
* *(Optional extras)* files in `data/modeling/static/extra/`

---

## Outputs

* **Static attributes (all basins):**
  `data/modeling/static/basin_attributes.parquet`
* **Provenance:**
  `data/modeling/static/basin_attributes.meta.json`

---

## Final schema (columns)

```
basin_id, basin_name,
longitude, latitude,
area_sqkm, perimeter_km, compactness, perim_area_ratio,
dem_min, dem_max, dem_mean, dem_median, dem_std, elev_range_m, dem_cv,
slope_mean, slope_p90_deg, relief_m,
log_acc_mean, log_acc_max,
pct_slope_gt_a, pct_slope_gt_b,
... (plus any additional columns merged from static/extra)
```

**Notes**

* `compactness = 4π · area_sqkm / perimeter_km²` (0–1; higher = more compact)
* `perim_area_ratio = perimeter_km / √area_sqkm` (elongation proxy)
* `elev_range_m = dem_max − dem_min`
* `dem_cv = dem_std / dem_mean` (guarded for zero mean)
* Accumulation fields stored as `log1p(*)` to stabilize scale.

---

## How this is used next

These static attributes are **left-joined** onto each `(basin_id, date_local)` row when building **daily dynamic features** (Stage D Part 2), then carried into the modeling datasets.


## Resolve Project Root

In [1]:
# === Resolve Project Root ===
from pathlib import Path
import subprocess

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p/"data").exists() and (p/"code").exists():
            return p
        if (p/".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/liuq13/bhutan_climate_modeling


In [2]:
# === Paths & settings ===
import pandas as pd, numpy as np, json, re
from datetime import datetime

SHAPE_CSV         = PROJECT_ROOT / "data/boundaries/processed/basin_shape_stats.csv"
DEM_ACC_CSV       = PROJECT_ROOT / "data/boundaries/processed/basin_dem_acc_stats.csv"
LOOKUP_CSV        = PROJECT_ROOT / "data/boundaries/processed/basin_lookup.csv"

OUT_DIR           = PROJECT_ROOT / "data/modeling/static"
OUT_PARQUET       = OUT_DIR / "basin_attributes.parquet"
OUT_META          = OUT_DIR / "basin_attributes.meta.json"

# Optional plug-in folder for future geospatial features you may add later
EXTRA_DIR         = OUT_DIR / "extra"     # drop CSV/Parquet here to auto-merge
ALLOW_OVERWRITE   = False                 # if an extra file has a column that already exists, append "_extra"

OUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# === Helpers ===
def basin_key_series(s: pd.Series) -> pd.Series:
    """Case/spacing-insensitive basin name key (handles underscores/spaces)."""
    return (s.astype(str)
             .str.strip()
             .str.replace(r"[_\s]+", " ", regex=True)
             .str.lower())

def first_existing(d: pd.DataFrame, names: list[str]) -> str:
    for n in names:
        if n in d.columns:
            return n
    raise KeyError(f"None of the expected columns {names} found in: {list(d.columns)}")

def compactness(area_sqkm: pd.Series, perimeter_km: pd.Series) -> pd.Series:
    # 4πA/P², unitless (consistent if A in km² and P in km)
    return (4*np.pi*area_sqkm) / np.where(perimeter_km>0, perimeter_km**2, np.nan)

def perim_area_ratio(perimeter_km: pd.Series, area_sqkm: pd.Series) -> pd.Series:
    return perimeter_km / np.sqrt(np.where(area_sqkm>0, area_sqkm, np.nan))


## Load & clean basin_shape_stats.csv

In [4]:
# === Load shape stats ===
shape = pd.read_csv(SHAPE_CSV)

# Flexible column discovery (your header preview showed "perimeter_km" or "perimeter_kr")
lon_col = first_existing(shape, ["longitude","lon","LONGITUDE"])
lat_col = first_existing(shape, ["latitude","lat","LATITUDE"])
area_col = first_existing(shape, ["area_sqkm","area_km2","AREA_SQKM"])
perim_col = first_existing(shape, ["perimeter_km","perimeter_kr","perimeter_kilometers","PERIMETER_KM"])
basin_name_col = first_existing(shape, ["basin","Basin","basin_name","BASIN_NAME"])

shape = shape.rename(columns={
    basin_name_col: "basin_name",
    lon_col: "longitude",
    lat_col: "latitude",
    area_col: "area_sqkm",
    perim_col: "perimeter_km",
})

# Keep only what we need; compute shape-derived features
shape = shape[["basin_name","longitude","latitude","area_sqkm","perimeter_km"]].copy()
shape["compactness"]      = compactness(shape["area_sqkm"], shape["perimeter_km"])
shape["perim_area_ratio"] = perim_area_ratio(shape["perimeter_km"], shape["area_sqkm"])

shape["basin_key"] = basin_key_series(shape["basin_name"])

# Check duplicates by key
dups = shape["basin_key"].value_counts()
if (dups>1).any():
    raise ValueError(f"Duplicate basins in shape stats for keys:\n{dups[dups>1]}")


In [9]:
shape

,basin_name,longitude,latitude,area_sqkm,perimeter_km,compactness,perim_area_ratio,basin_key
0,Aiechhu,90.464808,26.945874,1963.905886,0.002897,2.940004e+09,0.000065,aiechhu
1,Merak_Sakteng,92.033958,27.329946,138.810677,0.000645,4.188638e+09,0.000055,merak sakteng
2,Mangdechhu,90.649025,27.506149,7436.605311,0.004425,4.771824e+09,0.000051,mangdechhu
3,Jaldhakha,88.987850,27.061616,1038.942931,0.002133,2.869328e+09,0.000066,jaldhakha
4,Amochhu,89.120110,27.350269,3929.619367,0.004123,2.904399e+09,0.000066,amochhu
5,Wangchhu,89.482723,27.353115,4608.336971,0.004038,3.551970e+09,0.000059,wangchhu
6,Drangmechhu,91.395029,27.842435,21098.266822,0.009023,3.256464e+09,0.000062,drangmechhu
7,Punatsangchhu,89.939466,27.562875,9765.747380,0.005941,3.476406e+09,0.000060,punatsangchhu
8,Nyera_Amari,91.652809,26.987295,2262.233550,0.003570,2.230784e+09,0.000075,nyera amari
9,Jomori,91.959758,27.123484,750.319391,0.001549,3.931682e+09,0.000057,jomori


## Load & clean basin_dem_acc_stats.csv

In [7]:
# === Load DEM + accumulation stats ===
dem = pd.read_csv(DEM_ACC_CSV)

basin_name_col = first_existing(dem, ["basin","Basin","basin_name","BASIN_NAME"])
dem = dem.rename(columns={basin_name_col: "basin_name"})

# Flexible column picks
cols_expected = {
    "dem_min":      first_existing(dem, ["dem_min","DEM_MIN"]),
    "dem_max":      first_existing(dem, ["dem_max","DEM_MAX"]),
    "dem_mean":     first_existing(dem, ["dem_mean","DEM_MEAN"]),
    "dem_std":      first_existing(dem, ["dem_std","DEM_STD"]),
    "dem_median":   first_existing(dem, ["dem_median","DEM_MEDIAN"]),
    "relief_m":     first_existing(dem, ["relief_m","RELIEF_M"]),
    "slope_mean":   first_existing(dem, ["slope_mean","slope_mean_deg"]),
    "slope_p90_deg":first_existing(dem, ["slope_p90_deg","slope_p90","SLOPE_P90_DEG","SLOPE_P90"]),
}
# Optional accumulation & slope-fraction columns (best-judgment)
acc_mean_col = next((c for c in ["acc_mean","ACC_MEAN"] if c in dem.columns), None)
acc_max_col  = next((c for c in ["acc_max","ACC_MAX"]   if c in dem.columns), None)

# Two unknown slope % columns (keep if present)
pct_cols = [c for c in dem.columns if re.match(r"^pct_slope", c, flags=re.IGNORECASE)]
pct_cols = pct_cols[:2]  # keep at most two

# Build tight frame
keep = ["basin_name"] + list(cols_expected.values()) + [c for c in [acc_mean_col, acc_max_col] if c] + pct_cols
dem = dem[keep].copy()
dem = dem.rename(columns={v:k for k,v in cols_expected.items()})

if acc_mean_col: dem = dem.rename(columns={acc_mean_col:"acc_mean"})
if acc_max_col:  dem = dem.rename(columns={acc_max_col:"acc_max"})

# Derived
dem["elev_range_m"] = dem["dem_max"] - dem["dem_min"]
dem["dem_cv"]       = dem["dem_std"] / dem["dem_mean"].replace({0:np.nan})

# Stabilized logs for accumulation (only if available)
if "acc_mean" in dem.columns:
    dem["log_acc_mean"] = np.log1p(dem["acc_mean"].clip(lower=0))
if "acc_max" in dem.columns:
    dem["log_acc_max"]  = np.log1p(dem["acc_max"].clip(lower=0))

# Rename the two slope-fraction columns (if present) to generic names
if len(pct_cols) >= 1: dem = dem.rename(columns={pct_cols[0]: "pct_slope_gt_a"})
if len(pct_cols) >= 2: dem = dem.rename(columns={pct_cols[1]: "pct_slope_gt_b"})

dem["basin_key"] = basin_key_series(dem["basin_name"])

# Check duplicates
dups = dem["basin_key"].value_counts()
if (dups>1).any():
    raise ValueError(f"Duplicate basins in DEM/acc stats for keys:\n{dups[dups>1]}")


In [10]:
dem

,basin_name,dem_min,dem_max,dem_mean,dem_std,dem_median,relief_m,slope_mean,slope_p90_deg,acc_mean,acc_max,pct_slope_gt_a,pct_slope_gt_b,elev_range_m,dem_cv,log_acc_mean,log_acc_max,basin_key
0,Aiechhu,92.0,4160.0,1183.538707,699.525166,1091.0,4068.0,21.200437,34.513462,210.078824,3754703.0,72.055599,21.698405,4068.0,0.591045,5.352232,15.138520,aiechhu
1,Merak_Sakteng,2690.0,4483.0,3882.785851,340.278285,3965.0,1793.0,20.382105,31.582006,65.099067,10252.0,71.324777,13.740984,1793.0,0.087638,4.191155,9.235326,merak sakteng
2,Mangdechhu,109.0,7065.0,3230.588248,1302.878378,3287.0,6956.0,23.063430,35.971821,1070.502172,979449.0,77.342031,25.226937,6956.0,0.403294,6.976817,13.794746,mangdechhu
3,Jaldhakha,210.0,4577.0,1588.797296,1011.341252,1403.0,4367.0,21.281973,33.426529,199.115628,72476.0,74.670477,18.176001,4367.0,0.636545,5.298895,11.191025,jaldhakha
4,Amochhu,164.0,6689.0,3187.160344,1411.272228,3521.0,6525.0,22.034436,34.454346,894.656866,497461.0,75.496564,21.330364,6525.0,0.442799,6.797557,13.117274,amochhu
5,Wangchhu,117.0,6689.0,3228.431849,1081.635175,3270.0,6572.0,23.030790,34.844448,1076.646375,602706.0,80.520147,22.038662,6572.0,0.335034,6.982535,13.309186,wangchhu
6,Drangmechhu,90.0,7127.0,3824.023227,1417.195291,4292.0,7037.0,22.540061,35.928204,1944.081318,3756205.0,74.820652,24.171486,7037.0,0.370603,7.573059,15.138920,drangmechhu
7,Punatsangchhu,94.0,7087.0,3149.052890,1405.579514,3117.0,6993.0,23.648238,36.667324,1444.768641,1277715.0,79.314210,26.663519,6993.0,0.446350,7.276396,14.060585,punatsangchhu
8,Nyera_Amari,99.0,4462.0,1536.849713,1003.422671,1391.0,4363.0,22.749070,36.319275,313.954688,145267.0,75.249209,25.221410,4363.0,0.652909,5.752429,11.886336,nyera amari
9,Jomori,217.0,4477.0,2239.513228,918.928038,2243.0,4260.0,26.134153,37.785282,408.963529,95581.0,88.162490,33.790817,4260.0,0.410325,6.016068,11.467740,jomori


## Merge shape + DEM stats → attach basin_id

In [8]:
# === Merge shape + DEM/acc ===
static0 = shape.merge(dem, on=["basin_key","basin_name"], how="outer", validate="one_to_one")

# Attach numeric basin_id (authoritative join key downstream)
lookup = pd.read_csv(LOOKUP_CSV)
name_col = first_existing(lookup, ["basin_name","BASIN_NAME","basin"])
lookup = lookup.rename(columns={name_col: "basin_name"})
lookup["basin_key"] = basin_key_series(lookup["basin_name"])

static = static0.merge(lookup[["basin_key","basin_name","basin_id"]],
                       on="basin_key", how="left", suffixes=("", "_lkp"))

# Prefer lookup display name where available
static["basin_name"] = np.where(static["basin_name_lkp"].notna(), static["basin_name_lkp"], static["basin_name"])
static = static.drop(columns=["basin_name_lkp"])

# QA: report any missing basin_id
missing_id = static[static["basin_id"].isna()][["basin_name"]].sort_values("basin_name").drop_duplicates()
if len(missing_id):
    print("WARNING: These basins did not find a basin_id in lookup:")
    display(missing_id.head(20))


,basin_name
6,Merak_Sakteng


In [11]:
static

,basin_name,longitude,latitude,area_sqkm,perimeter_km,compactness,perim_area_ratio,basin_key,dem_min,dem_max,...,slope_p90_deg,acc_mean,acc_max,pct_slope_gt_a,pct_slope_gt_b,elev_range_m,dem_cv,log_acc_mean,log_acc_max,basin_id
0,Aiechhu,90.464808,26.945874,1963.905886,0.002897,2.940004e+09,0.000065,aiechhu,92.0,4160.0,...,34.513462,210.078824,3754703.0,72.055599,21.698405,4068.0,0.591045,5.352232,15.138520,1.0
1,Amochhu,89.120110,27.350269,3929.619367,0.004123,2.904399e+09,0.000066,amochhu,164.0,6689.0,...,34.454346,894.656866,497461.0,75.496564,21.330364,6525.0,0.442799,6.797557,13.117274,5.0
2,Drangmechhu,91.395029,27.842435,21098.266822,0.009023,3.256464e+09,0.000062,drangmechhu,90.0,7127.0,...,35.928204,1944.081318,3756205.0,74.820652,24.171486,7037.0,0.370603,7.573059,15.138920,7.0
3,Jaldhakha,88.987850,27.061616,1038.942931,0.002133,2.869328e+09,0.000066,jaldhakha,210.0,4577.0,...,33.426529,199.115628,72476.0,74.670477,18.176001,4367.0,0.636545,5.298895,11.191025,4.0
4,Jomori,91.959758,27.123484,750.319391,0.001549,3.931682e+09,0.000057,jomori,217.0,4477.0,...,37.785282,408.963529,95581.0,88.162490,33.790817,4260.0,0.410325,6.016068,11.467740,10.0
5,Mangdechhu,90.649025,27.506149,7436.605311,0.004425,4.771824e+09,0.000051,mangdechhu,109.0,7065.0,...,35.971821,1070.502172,979449.0,77.342031,25.226937,6956.0,0.403294,6.976817,13.794746,3.0
6,Merak_Sakteng,92.033958,27.329946,138.810677,0.000645,4.188638e+09,0.000055,merak sakteng,2690.0,4483.0,...,31.582006,65.099067,10252.0,71.324777,13.740984,1793.0,0.087638,4.191155,9.235326,NaN
7,Nyera_Amari,91.652809,26.987295,2262.233550,0.003570,2.230784e+09,0.000075,nyera amari,99.0,4462.0,...,36.319275,313.954688,145267.0,75.249209,25.221410,4363.0,0.652909,5.752429,11.886336,9.0
8,Punatsangchhu,89.939466,27.562875,9765.747380,0.005941,3.476406e+09,0.000060,punatsangchhu,94.0,7087.0,...,36.667324,1444.768641,1277715.0,79.314210,26.663519,6993.0,0.446350,7.276396,14.060585,8.0
9,Wangchhu,89.482723,27.353115,4608.336971,0.004038,3.551970e+09,0.000059,wangchhu,117.0,6689.0,...,34.844448,1076.646375,602706.0,80.520147,22.038662,6572.0,0.335034,6.982535,13.309186,6.0


## Optional: auto-merge any future extras you drop into static/extra/

In [12]:
# === Auto-merge future extras from data/modeling/static/extra ===
added_from_extra = []

if EXTRA_DIR.exists():
    for p in sorted(EXTRA_DIR.glob("*")):
        if not p.is_file() or p.suffix.lower() not in {".csv",".parquet"}:
            continue
        try:
            df = pd.read_parquet(p) if p.suffix.lower()==".parquet" else pd.read_csv(p)
        except Exception as e:
            print(f"[SKIP] {p.name}: could not read ({e})")
            continue

        cols_before = set(static.columns)
        if "basin_id" in df.columns:
            merged = static.merge(df, on="basin_id", how="left", suffixes=("", "_extra"))
        else:
            # try basin_name key
            if "basin_name" not in df.columns and "basin" in df.columns:
                df = df.rename(columns={"basin":"basin_name"})
            if "basin_name" not in df.columns:
                print(f"[SKIP] {p.name}: needs basin_id or basin_name column")
                continue
            df["basin_key"] = basin_key_series(df["basin_name"])
            merged = static.merge(df.drop(columns=["basin_name"], errors="ignore"),
                                  on="basin_key", how="left", suffixes=("", "_extra"))
        # Handle collisions
        if not ALLOW_OVERWRITE:
            collisions = [c for c in df.columns if c in static.columns and c not in ("basin_id","basin_name","basin_key")]
            for c in collisions:
                if c in merged.columns and c+"_extra" in merged.columns:
                    merged.drop(columns=[c], inplace=True)
                    merged.rename(columns={c+"_extra": c+"_extra"}, inplace=True)
        static = merged
        new_cols = [c for c in static.columns if c not in cols_before]
        added_from_extra.append({"file": str(p.relative_to(PROJECT_ROOT)), "added_columns": new_cols})

if added_from_extra:
    print("Merged extra static sources:")
    for item in added_from_extra:
        print("  •", item["file"], "→", ", ".join(item["added_columns"]))


## Reorder/limit columns, write outputs, and a quick preview

In [13]:
# === Column order & write ===
base_cols = [
    "basin_id","basin_name",
    "longitude","latitude",
    "area_sqkm","perimeter_km","compactness","perim_area_ratio",
    "dem_min","dem_max","dem_mean","dem_median","dem_std","elev_range_m","dem_cv",
    "slope_mean","slope_p90_deg","relief_m",
    "log_acc_mean","log_acc_max",
    "pct_slope_gt_a","pct_slope_gt_b",
]

# Keep existing ones in order, then append any extras at the end
ordered = [c for c in base_cols if c in static.columns]
extra   = [c for c in static.columns if c not in ordered + ["basin_key"]]
final_df = static[ordered + extra].copy()

# Persist
final_df = final_df.sort_values(["basin_id","basin_name"], na_position="last").reset_index(drop=True)
final_df.to_parquet(OUT_PARQUET, index=False)

meta = {
    "created_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "source_files": [str(SHAPE_CSV.relative_to(PROJECT_ROOT)),
                     str(DEM_ACC_CSV.relative_to(PROJECT_ROOT)),
                     str(LOOKUP_CSV.relative_to(PROJECT_ROOT))],
    "extra_sources": added_from_extra,
    "n_basins": int(final_df["basin_id"].notna().sum()),
    "columns": list(final_df.columns),
}
with open(OUT_META, "w") as f:
    json.dump(meta, f, indent=2)

print("Wrote:", OUT_PARQUET)
print("Wrote:", OUT_META)
display(final_df.head(10))


Wrote: /Users/liuq13/bhutan_climate_modeling/data/modeling/static/basin_attributes.parquet
Wrote: /Users/liuq13/bhutan_climate_modeling/data/modeling/static/basin_attributes.meta.json


,basin_id,basin_name,longitude,latitude,area_sqkm,perimeter_km,compactness,perim_area_ratio,dem_min,dem_max,...,dem_cv,slope_mean,slope_p90_deg,relief_m,log_acc_mean,log_acc_max,pct_slope_gt_a,pct_slope_gt_b,acc_mean,acc_max
0,1.0,Aiechhu,90.464808,26.945874,1963.905886,0.002897,2.940004e+09,0.000065,92.0,4160.0,...,0.591045,21.200437,34.513462,4068.0,5.352232,15.138520,72.055599,21.698405,210.078824,3754703.0
1,3.0,Mangdechhu,90.649025,27.506149,7436.605311,0.004425,4.771824e+09,0.000051,109.0,7065.0,...,0.403294,23.063430,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0
2,4.0,Jaldhakha,88.987850,27.061616,1038.942931,0.002133,2.869328e+09,0.000066,210.0,4577.0,...,0.636545,21.281973,33.426529,4367.0,5.298895,11.191025,74.670477,18.176001,199.115628,72476.0
3,5.0,Amochhu,89.120110,27.350269,3929.619367,0.004123,2.904399e+09,0.000066,164.0,6689.0,...,0.442799,22.034436,34.454346,6525.0,6.797557,13.117274,75.496564,21.330364,894.656866,497461.0
4,6.0,Wangchhu,89.482723,27.353115,4608.336971,0.004038,3.551970e+09,0.000059,117.0,6689.0,...,0.335034,23.030790,34.844448,6572.0,6.982535,13.309186,80.520147,22.038662,1076.646375,602706.0
5,7.0,Drangmechhu,91.395029,27.842435,21098.266822,0.009023,3.256464e+09,0.000062,90.0,7127.0,...,0.370603,22.540061,35.928204,7037.0,7.573059,15.138920,74.820652,24.171486,1944.081318,3756205.0
6,8.0,Punatsangchhu,89.939466,27.562875,9765.747380,0.005941,3.476406e+09,0.000060,94.0,7087.0,...,0.446350,23.648238,36.667324,6993.0,7.276396,14.060585,79.314210,26.663519,1444.768641,1277715.0
7,9.0,Nyera_Amari,91.652809,26.987295,2262.233550,0.003570,2.230784e+09,0.000075,99.0,4462.0,...,0.652909,22.749070,36.319275,4363.0,5.752429,11.886336,75.249209,25.221410,313.954688,145267.0
8,10.0,Jomori,91.959758,27.123484,750.319391,0.001549,3.931682e+09,0.000057,217.0,4477.0,...,0.410325,26.134153,37.785282,4260.0,6.016068,11.467740,88.162490,33.790817,408.963529,95581.0
9,NaN,Merak_Sakteng,92.033958,27.329946,138.810677,0.000645,4.188638e+09,0.000055,2690.0,4483.0,...,0.087638,20.382105,31.582006,1793.0,4.191155,9.235326,71.324777,13.740984,65.099067,10252.0
